# 02 - Model and detector comparison

Reads the tables produced by `python run_pipeline.py`, so it runs in a second and always
shows the same numbers as `PROJECT_GUIDE.md` and the dashboard.

Run `python run_pipeline.py` first.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from pathlib import Path

pd.set_option("display.width", 200)
REPORTS = Path("..") / "reports"

metrics = pd.read_csv(REPORTS / "forecast_metrics.csv")
print("models:", metrics["model"].unique().tolist())

models: ['snaive_24', 'snaive_168', 'ridge', 'random_forest', 'lightgbm']


In [2]:
# Forecast accuracy: one table, every model, test fold, averaged over buildings.
test = metrics[metrics["fold"] == "test"]
test.groupby("model")[["mae", "rmse", "cv_rmse_pct", "nmbe_pct", "smape_pct"]].mean().round(2)

,mae,rmse,cv_rmse_pct,nmbe_pct,smape_pct
model,,,,,
lightgbm,10.37,16.26,20.77,6.47,16.19
random_forest,9.60,15.54,19.32,4.37,15.18
ridge,10.53,16.55,18.97,2.20,14.24
snaive_168,11.83,20.89,24.49,-0.68,14.40
snaive_24,13.97,23.47,25.99,-0.53,16.45


In [3]:
# Where does the best model fail? CV(RMSE) by building shows the scale trap:
# a tiny auditorium has a large percentage error even with a small absolute one.
best = "random_forest"
test[test["model"] == best][["building_label", "mae", "rmse", "cv_rmse_pct", "nmbe_pct"]].round(2)

,building_label,mae,rmse,cv_rmse_pct,nmbe_pct
8,Andre (Laboratory),16.53,28.55,6.01,0.79
18,Elvira (Student Union),10.39,16.69,9.69,-0.29
28,Claire (Classroom),5.59,9.26,14.62,-0.80
38,Kendrick (Auditorium),5.90,7.65,46.96,17.78


In [4]:
# Anomaly detection on injected faults: precision/recall/F1 per method.
scores = pd.read_csv(REPORTS / "detection_scores.csv")
overall = scores[scores["anomaly_type"] == "ALL"]
overall.groupby("method")[["precision", "recall", "f1", "false_alarm_rate_pct"]].mean().round(3)

,precision,recall,f1,false_alarm_rate_pct
method,,,,
Adaptive hybrid (30-day recalibration),0.732,0.166,0.262,2.840
Hybrid (persistence + flatline),0.454,0.350,0.338,20.335
Isolation Forest,0.236,0.343,0.257,35.388
Residual (robust z),0.346,0.375,0.315,25.288
Residual + persistence,0.364,0.312,0.291,20.230


In [5]:
# Recall per anomaly type: what each detector is actually good at.
per_type = scores[scores["anomaly_type"] != "ALL"]
per_type.pivot_table(index="anomaly_type", columns="method", values="recall", aggfunc="mean").round(3)

method,Adaptive hybrid (30-day recalibration),Hybrid (persistence + flatline),Isolation Forest,Residual (robust z),Residual + persistence
anomaly_type,,,,,
after_hours,0.383,0.638,0.770,0.714,0.638
drift,0.074,0.250,0.241,0.308,0.250
flatline,0.439,0.622,0.486,0.428,0.392
spike,0.250,0.389,0.889,0.861,0.389


In [6]:
# Sensitivity: the precision/recall trade-off controlled by the z threshold.
sensitivity = pd.read_csv(REPORTS / "detection_sensitivity.csv")
(
    sensitivity.groupby(["z_threshold", "method"])[["precision", "recall", "f1", "false_alarm_rate_pct"]]
    .mean()
    .round(3)
)

precision  recall     f1  false_alarm_rate_pct
z_threshold method                                                              
3.0         adaptive_hybrid           0.732   0.166  0.262                 2.840
            hybrid                    0.454   0.350  0.338                20.335
            isolation_forest          0.236   0.343  0.257                35.388
            residual                  0.346   0.375  0.315                25.288
            residual+persistence      0.364   0.312  0.291                20.230
3.5         adaptive_hybrid           0.748   0.143  0.235                 2.280
            hybrid                    0.508   0.294  0.312                16.267
            isolation_forest          0.236   0.343  0.257                35.388
            residual                  0.363   0.314  0.288                20.868
            residual+persistence      0.384   0.255  0.261                16.130
4.0         adaptive_hybrid           0.767   0.126  0.214                 1.972
            hybrid                    0.556   0.254  0.291                13.138
            isolation_forest          0.236   0.343  0.257                35.388
            residual                  0.375   0.270  0.267                17.012
            residual+persistence      0.372   0.214  0.235                13.000
4.5         adaptive_hybrid           0.766   0.111  0.191                 1.730
            hybrid                    0.608   0.225  0.276                10.482
            isolation_forest          0.236   0.343  0.257                35.388
            residual                  0.411   0.235  0.252                13.942
            residual+persistence      0.410   0.182  0.216                10.330
5.0         adaptive_hybrid           0.772   0.100  0.176                 1.488
            hybrid                    0.658   0.206  0.267                 8.382
            isolation_forest          0.236   0.343  0.257                35.388
            residual                  0.486   0.207  0.235                11.358
            residual+persistence      0.466   0.159  0.199                 8.200
6.0         adaptive_hybrid           0.800   0.087  0.156                 1.138
            hybrid                    0.699   0.178  0.252                 5.922
            isolation_forest          0.236   0.343  0.257                35.388
            residual                  0.575   0.170  0.215                 8.125
            residual+persistence      0.558   0.128  0.179                 5.650